In [2]:
import pandas as pd

# Load the weather data that we already collected
weather_df = pd.read_csv(
    "../data/cleaned/weather_complete.csv"
)

print("Rows:", len(weather_df))
print("Unique destinations:", weather_df["destination"].nunique())

display(weather_df.head())

Rows: 9
Unique destinations: 9


,destination,country,latitude,longitude,temperature,feels_like,humidity,pressure,wind_speed,cloudiness,weather_condition,weather_description,visibility,rain_1h,timestamp
0,Hyderabad,IN,17.3753,78.4744,26.23,26.23,73,1010,5.14,22,Clouds,few clouds,10000.0,0.00,2026-08-25T14:29:49+00:00
1,Goa,IN,15.3333,74.0833,26.21,26.21,92,1010,1.96,66,Rain,light rain,10000.0,0.23,2026-08-25T14:28:43+00:00
2,Munnar,IN,10.1000,77.0667,16.23,16.42,96,1016,2.11,100,Clouds,overcast clouds,NaN,0.00,2026-08-25T14:28:58+00:00
3,Manali,IN,13.1667,80.2667,30.29,37.29,79,1007,0.45,98,Clouds,overcast clouds,10000.0,0.00,2026-08-25T14:30:02+00:00
4,Jaipur,IN,26.9167,75.8167,27.62,32.42,89,1004,0.00,97,Clouds,overcast clouds,10000.0,0.00,2026-08-25T14:26:24+00:00


In [3]:
# Load the 50-destination master list

destination_master = pd.read_csv(
    "../data/raw/destination_master.csv"
)

# Find destinations for which weather data has not been collected yet
missing_weather_destinations = destination_master[
    ~destination_master["destination"].isin(
        weather_df["destination"]
    )
]["destination"].tolist()

print("Total destinations:", len(destination_master))
print("Weather already collected:", weather_df["destination"].nunique())
print("Weather missing:", len(missing_weather_destinations))

print("\nMissing weather destinations:")
print(missing_weather_destinations)

Total destinations: 50
Weather already collected: 9
Weather missing: 41

Missing weather destinations:
['Udaipur', 'Jaisalmer', 'Jodhpur', 'Agra', 'Varanasi', 'Rishikesh', 'Shimla', 'Mussoorie', 'Nainital', 'Darjeeling', 'Gangtok', 'Ooty', 'Coorg', 'Wayanad', 'Alappuzha', 'Kochi', 'Varkala', 'Mahabalipuram', 'Hampi', 'Mysore', 'Gokarna', 'Andaman', 'Mumbai', 'Delhi', 'Amritsar', 'Ladakh', 'Srinagar', 'Dharamshala', 'Kolkata', 'Bengaluru', 'Chennai', 'Pune', 'Ahmedabad', 'Bhopal', 'Indore', 'Ranchi', 'Bhubaneswar', 'Shillong', 'Kaziranga', 'Jim Corbett', 'Ranthambore']


In [5]:
import requests
from datetime import datetime, timezone
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Get the OpenWeather API key
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

print("API key loaded:", WEATHER_API_KEY is not None)

API key loaded: True


In [7]:
def get_weather(city):
    """
    Fetch current weather information for a given city.

    Parameters
    ----------
    city : str
        Name of the destination city.

    Returns
    -------
    dict or None
        A cleaned dictionary containing the weather features
        required for our Travel Agent project.
        Returns None if the API request fails.
    """

    # OpenWeather current weather endpoint
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Parameters required by the API
    params = {
        "q": city,
        "appid": WEATHER_API_KEY,
        "units": "metric"
    }

    try:
        # Send request to OpenWeather
        response = requests.get(url, params=params, timeout=10)

        # Raise an exception automatically for HTTP errors
        response.raise_for_status()

        # Convert JSON response into a Python dictionary
        data = response.json()

        # Extract and return only the fields needed by our project
        weather = {
            "destination": data["name"],
            "country": data["sys"]["country"],
            "latitude": data["coord"]["lat"],
            "longitude": data["coord"]["lon"],
            "temperature": data["main"]["temp"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "pressure": data["main"]["pressure"],
            "wind_speed": data["wind"]["speed"],
            "cloudiness": data["clouds"]["all"],
            "weather_condition": data["weather"][0]["main"],
            "weather_description": data["weather"][0]["description"],
            "visibility": data.get("visibility"),
            "rain_1h": data.get("rain", {}).get("1h", 0),
            # Convert Unix timestamp returned by OpenWeather
            # into a readable UTC datetime.
            "timestamp": datetime.fromtimestamp(
                data["dt"],
                tz=timezone.utc
            ).isoformat()
        }

        return weather

    except requests.exceptions.RequestException as e:
        # Handle connection, timeout, HTTP, and API errors
        print(f"API request failed for {city}: {e}")
        return None

    except KeyError as e:
        # Handle unexpected/missing fields in the API response
        print(f"Unexpected API response for {city}. Missing field: {e}")
        return None

In [8]:
# Test the existing weather function with one missing destination

test_weather = get_weather("Udaipur")

print(test_weather)

{'destination': 'Udaipur', 'country': 'IN', 'latitude': 24.5712, 'longitude': 73.6918, 'temperature': 28.5, 'feels_like': 31.05, 'humidity': 66, 'pressure': 1006, 'wind_speed': 4.79, 'cloudiness': 100, 'weather_condition': 'Clouds', 'weather_description': 'overcast clouds', 'visibility': 10000, 'rain_1h': 0, 'timestamp': '2026-08-27T12:00:06+00:00'}


In [9]:
# Collect weather for a small batch of missing destinations.
# We start with only 5 to avoid exhausting the API quota again.

weather_records_new = []

for city in missing_weather_destinations[:5]:

    print("=" * 50)
    print(f"Collecting weather: {city}")

    weather = get_weather(city)

    if weather is not None:
        weather_records_new.append(weather)
        print("Success")
    else:
        print("Failed")

print("=" * 50)
print("New weather records:", len(weather_records_new))

Success
Success
Success
Success
Success
New weather records: 5


In [10]:
# Collect weather for all remaining destinations.
# The first 5 missing destinations were already tested successfully,
# so we start from index 5.

remaining_weather_destinations = missing_weather_destinations[5:]

weather_records_remaining = []

print("Destinations remaining:", len(remaining_weather_destinations))

for city in remaining_weather_destinations:

    print("=" * 50)
    print(f"Collecting weather: {city}")

    weather = get_weather(city)

    if weather is not None:
        weather_records_remaining.append(weather)
        print("Success")
    else:
        print("Failed")

print("=" * 50)
print("New weather records:", len(weather_records_remaining))

In [13]:
# Combine the previously collected weather data with the new records

new_weather_df = pd.DataFrame(weather_records_remaining)

combined_weather_df = pd.concat(
    [weather_df, new_weather_df],
    ignore_index=True
)

# Remove duplicate destinations, keeping the latest collected record
combined_weather_df = (
    combined_weather_df
    .drop_duplicates(
        subset=["destination"],
        keep="last"
    )
)

print("Total weather records:", len(combined_weather_df))
print("Unique destinations:", combined_weather_df["destination"].nunique())

print("\nDestinations collected:")
print(sorted(combined_weather_df["destination"].tolist()))

Total weather records: 38
Unique destinations: 38

Destinations collected:
['Ahmedabad', 'Alappuzha', 'Amritsar', 'Andaman', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Darjeeling', 'Delhi', 'Gangtok', 'Goa', 'Gokarna', 'Hampi', 'Hyderabad', 'Indore', 'Jaipur', 'Kochi', 'Kodaikanal', 'Kolkata', 'Mahabalipuram', 'Manali', 'Mumbai', 'Munnar', 'Mussoorie', 'Mysore', 'Nainital', 'Ooty', 'Pahalgam', 'Pondicherry', 'Pune', 'Ranchi', 'Rishikesh', 'Shillong', 'Shimla', 'Srinagar', 'Thiruvananthapuram', 'Varkala']


In [14]:
# Find the destinations that still don't have weather data

remaining_missing_weather = destination_master[
    ~destination_master["destination"].isin(
        combined_weather_df["destination"]
    )
]["destination"].tolist()

print("Weather collected:", len(combined_weather_df))
print("Weather still missing:", len(remaining_missing_weather))

print("\nRemaining destinations:")
print(remaining_missing_weather)

Weather collected: 38
Weather still missing: 12

Remaining destinations:
['Udaipur', 'Jaisalmer', 'Jodhpur', 'Agra', 'Varanasi', 'Coorg', 'Wayanad', 'Ladakh', 'Dharamshala', 'Kaziranga', 'Jim Corbett', 'Ranthambore']


In [15]:
# Convert the 5 successfully tested weather records into a DataFrame

weather_test_df = pd.DataFrame(weather_records_new)

print("Test weather records:", len(weather_test_df))

display(
    weather_test_df[
        ["destination", "country", "temperature", "weather_condition"]
    ]
)

Test weather records: 5


,destination,country,temperature,weather_condition
0,Udaipur,IN,28.50,Clouds
1,Jaisalmer,IN,37.90,Clear
2,Jodhpur,IN,35.92,Clouds
3,Agra,IN,28.03,Clouds
4,Varanasi,IN,30.05,Clouds


In [16]:
# Combine all weather records collected so far

combined_weather_df = pd.concat(
    [
        weather_df,
        weather_test_df,
        new_weather_df
    ],
    ignore_index=True
)

# Keep only one record per destination.
# If a destination was collected more than once, keep the latest record.
combined_weather_df = (
    combined_weather_df
    .drop_duplicates(
        subset=["destination"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("Total weather records:", len(combined_weather_df))
print(
    "Unique destinations:",
    combined_weather_df["destination"].nunique()
)

Total weather records: 43
Unique destinations: 43


In [17]:
# Compare our collected weather destinations with the full destination list

remaining_missing_weather = destination_master[
    ~destination_master["destination"].isin(
        combined_weather_df["destination"]
    )
]["destination"].tolist()

print("Weather collected:", len(combined_weather_df))
print("Weather still missing:", len(remaining_missing_weather))

print("\nRemaining destinations:")
print(remaining_missing_weather)

Weather collected: 43
Weather still missing: 7

Remaining destinations:
['Coorg', 'Wayanad', 'Ladakh', 'Dharamshala', 'Kaziranga', 'Jim Corbett', 'Ranthambore']


In [18]:
# Collect weather for the final 7 destinations.
# These are the only destinations still missing weather data.

final_weather_records = []

for city in remaining_missing_weather:

    print("=" * 50)
    print(f"Collecting weather: {city}")

    weather = get_weather(city)

    if weather is not None:
        final_weather_records.append(weather)
        print("Success")
    else:
        print("Failed")

print("=" * 50)
print("Final batch records:", len(final_weather_records))

API request failed for Coorg: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Coorg&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Failed
API request failed for Wayanad: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Wayanad&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Failed
API request failed for Ladakh: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Ladakh&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Failed
API request failed for Dharamshala: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Dharamshala&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Failed
API request failed for Kaziranga: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Kaziranga&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Failed
API request failed for Jim Corbett: 404 Client Error: Not Found 

In [19]:
def get_weather_by_coordinates(city, latitude, longitude):
    """
    Fetch current weather using latitude and longitude.

    Using coordinates is more reliable than searching by city name,
    especially for destinations such as wildlife areas and regions
    that may not exist as exact city names in OpenWeather.
    """

    # OpenWeather current weather endpoint
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Use coordinates instead of the destination name
    params = {
        "lat": latitude,
        "lon": longitude,
        "appid": WEATHER_API_KEY,
        "units": "metric"
    }

    try:
        # Send the API request
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        # Raise an error if the request failed
        response.raise_for_status()

        # Convert API response to dictionary
        data = response.json()

        # Extract only the fields required by our project
        weather = {
            "destination": city,
            "country": data["sys"]["country"],
            "latitude": data["coord"]["lat"],
            "longitude": data["coord"]["lon"],
            "temperature": data["main"]["temp"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "pressure": data["main"]["pressure"],
            "wind_speed": data["wind"]["speed"],
            "cloudiness": data["clouds"]["all"],
            "weather_condition": data["weather"][0]["main"],
            "weather_description": data["weather"][0]["description"],
            "visibility": data.get("visibility"),
            "rain_1h": data.get("rain", {}).get("1h", 0),
            "timestamp": datetime.fromtimestamp(
                data["dt"],
                tz=timezone.utc
            ).isoformat()
        }

        return weather

    except requests.exceptions.RequestException as e:
        print(f"API request failed for {city}: {e}")
        return None

    except KeyError as e:
        print(
            f"Unexpected API response for {city}. "
            f"Missing field: {e}"
        )
        return None

In [21]:
# Load the existing Places dataset.
# This is local data, so no API request is made.

places_df = pd.read_csv(
    "../data/cleaned/places_cleaned.csv"
)

print("Rows:", len(places_df))
print("Columns:", places_df.columns.tolist())

Rows: 8144
Columns: ['query_category', 'search_radius_km', 'name', 'country', 'state', 'city', 'latitude', 'longitude', 'categories', 'formatted_address', 'place_id', 'limit_reached', 'destination_id', 'destination']


In [22]:
# Keep one latitude/longitude pair for each destination.

destination_coordinates = (
    places_df[
        ["destination", "latitude", "longitude"]
    ]
    .dropna()
    .drop_duplicates("destination")
)

print(
    "Destinations with coordinates:",
    len(destination_coordinates)
)

Destinations with coordinates: 50


In [23]:
# Select coordinates for the destinations whose
# city-name weather searches returned 404.

final_weather_locations = destination_coordinates[
    destination_coordinates["destination"].isin(
        remaining_missing_weather
    )
].copy()

print(
    "Final destinations found:",
    len(final_weather_locations)
)

display(final_weather_locations)

Final destinations found: 7


,destination,latitude,longitude
2410,Coorg,12.317029,75.702543
2416,Wayanad,11.660211,76.250743
5249,Ladakh,34.071584,77.632917
5502,Dharamshala,32.087091,76.254332
7972,Kaziranga,26.588174,93.401822
8079,Jim Corbett,29.557475,78.842495
8103,Ranthambore,26.020386,76.454990


In [24]:
# Collect weather for the final 7 destinations using coordinates.
# Coordinates are more reliable for regions and national parks
# that OpenWeather may not recognize by their destination name.

final_weather_records = []

for _, row in final_weather_locations.iterrows():

    city = row["destination"]
    latitude = row["latitude"]
    longitude = row["longitude"]

    print("=" * 50)
    print(f"Collecting weather: {city}")
    print(f"Coordinates: {latitude}, {longitude}")

    weather = get_weather_by_coordinates(
        city=city,
        latitude=latitude,
        longitude=longitude
    )

    if weather is not None:
        final_weather_records.append(weather)
        print("Success")
    else:
        print("Failed")

print("=" * 50)
print("Final weather records:", len(final_weather_records))

Coordinates: 12.3170292, 75.7025429
Success
Coordinates: 11.6602112, 76.2507426
Success
Coordinates: 34.0715843, 77.6329172
Success
Coordinates: 32.08709098397102, 76.25433199705348
Success
Coordinates: 26.588173984133707, 93.40182185554352
Success
Coordinates: 29.55747519978108, 78.84249502753302
Success
Coordinates: 26.0203856, 76.4549898
Success
Final weather records: 7


In [25]:
# Convert the final successful weather records into a DataFrame

final_weather_df = pd.DataFrame(final_weather_records)

print("Final batch rows:", len(final_weather_df))

display(
    final_weather_df[
        [
            "destination",
            "country",
            "latitude",
            "longitude",
            "temperature",
            "weather_condition"
        ]
    ]
)

Final batch rows: 7


,destination,country,latitude,longitude,temperature,weather_condition
0,Coorg,IN,12.3170,75.7025,22.76,Rain
1,Wayanad,IN,11.6602,76.2507,24.45,Rain
2,Ladakh,IN,34.0716,77.6329,24.92,Clouds
3,Dharamshala,IN,32.0871,76.2543,29.57,Clouds
4,Kaziranga,IN,26.5882,93.4018,28.92,Clouds
5,Jim Corbett,IN,29.5575,78.8425,29.80,Clouds
6,Ranthambore,IN,26.0204,76.4550,27.91,Rain


In [26]:
# Combine the existing 43 weather records with the final 7

combined_weather_df = pd.concat(
    [
        combined_weather_df,
        final_weather_df
    ],
    ignore_index=True
)

# Make sure there is exactly one weather record per destination
combined_weather_df = (
    combined_weather_df
    .drop_duplicates(
        subset=["destination"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("Total weather records:", len(combined_weather_df))
print(
    "Unique destinations:",
    combined_weather_df["destination"].nunique()
)

Total weather records: 50
Unique destinations: 50


In [27]:
# Inspect the Andaman weather record before correcting it

display(
    combined_weather_df[
        combined_weather_df["destination"] == "Andaman"
    ]
)

,destination,country,latitude,longitude,temperature,feels_like,humidity,pressure,wind_speed,cloudiness,weather_condition,weather_description,visibility,rain_1h,timestamp
28,Andaman,ID,-3.198,114.4621,26.79,27.82,60,1012,2.33,64,Clouds,broken clouds,10000.0,0.0,2026-08-27T12:11:42+00:00


In [28]:
# Get the correct coordinates for Andaman
# from our existing Places dataset.

andaman_coordinates = destination_coordinates[
    destination_coordinates["destination"] == "Andaman"
].iloc[0]

print("Correct Andaman coordinates:")
print("Latitude :", andaman_coordinates["latitude"])
print("Longitude:", andaman_coordinates["longitude"])

Correct Andaman coordinates:
Latitude : 12.757423949653935
Longitude: 92.84122300891394


In [29]:
# Fetch weather for Andaman using the verified coordinates.
# This avoids the incorrect "Andaman" city-name match.

correct_andaman_weather = get_weather_by_coordinates(
    city="Andaman",
    latitude=andaman_coordinates["latitude"],
    longitude=andaman_coordinates["longitude"]
)

print(correct_andaman_weather)


{'destination': 'Andaman', 'country': 'IN', 'latitude': 12.7574, 'longitude': 92.8412, 'temperature': 26.84, 'feels_like': 30, 'humidity': 87, 'pressure': 1009, 'wind_speed': 7.98, 'cloudiness': 100, 'weather_condition': 'Rain', 'weather_description': 'light rain', 'visibility': 10000, 'rain_1h': 0.47, 'timestamp': '2026-08-27T12:22:29+00:00'}


In [30]:
# Convert the corrected Andaman result into a one-row DataFrame

correct_andaman_df = pd.DataFrame(
    [correct_andaman_weather]
)

# Remove the incorrect Andaman record
combined_weather_df = combined_weather_df[
    combined_weather_df["destination"] != "Andaman"
]

# Add the corrected Andaman record
combined_weather_df = pd.concat(
    [
        combined_weather_df,
        correct_andaman_df
    ],
    ignore_index=True
)

# Keep exactly one record per destination
combined_weather_df = (
    combined_weather_df
    .drop_duplicates(
        subset=["destination"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("Total weather records:", len(combined_weather_df))
print(
    "Unique destinations:",
    combined_weather_df["destination"].nunique()
)

Total weather records: 50
Unique destinations: 50


In [31]:
# Check that every destination has complete weather data

weather_columns = [
    "destination",
    "country",
    "latitude",
    "longitude",
    "temperature",
    "feels_like",
    "humidity",
    "pressure",
    "wind_speed",
    "cloudiness",
    "weather_condition",
    "weather_description",
    "visibility",
    "rain_1h",
    "timestamp"
]

print("Missing values:")
print(
    combined_weather_df[weather_columns]
    .isna()
    .sum()
)

print("\nDuplicate destinations:")
print(
    combined_weather_df["destination"]
    .duplicated()
    .sum()
)

Missing values:
destination            0
country                0
latitude               0
longitude              0
temperature            0
feels_like             0
humidity               0
pressure               0
wind_speed             0
cloudiness             0
weather_condition      0
weather_description    0
visibility             1
rain_1h                0
timestamp              0
dtype: int64

Duplicate destinations:
0


In [32]:
# Find the destination where visibility is missing

display(
    combined_weather_df[
        combined_weather_df["visibility"].isna()
    ][
        ["destination", "country", "temperature", "weather_condition"]
    ]
)

,destination,country,temperature,weather_condition
2,Munnar,IN,16.23,Clouds


In [33]:
# Fill the single missing visibility value using the median
# of the successfully collected visibility values.

visibility_median = combined_weather_df["visibility"].median()

print("Visibility median:", visibility_median)

combined_weather_df["visibility"] = (
    combined_weather_df["visibility"]
    .fillna(visibility_median)
)

print(
    "Missing visibility after filling:",
    combined_weather_df["visibility"].isna().sum()
)

Visibility median: 10000.0
Missing visibility after filling: 0


In [34]:
# Final validation before saving the weather dataset

print("Rows:", len(combined_weather_df))
print(
    "Unique destinations:",
    combined_weather_df["destination"].nunique()
)

print("\nTotal missing values:")
print(combined_weather_df.isna().sum().sum())

print("\nDuplicate destinations:")
print(
    combined_weather_df["destination"].duplicated().sum()
)

Rows: 50
Unique destinations: 50

Total missing values:
0

Duplicate destinations:
0


In [35]:
# Save the completed weather dataset.
# This file will be loaded later by 04_Data_Integration.ipynb.

weather_output_path = "../data/cleaned/weather_data.csv"

combined_weather_df.to_csv(
    weather_output_path,
    index=False
)

print(f"Weather data saved to: {weather_output_path}")
print("Rows:", len(combined_weather_df))
print(
    "Unique destinations:",
    combined_weather_df["destination"].nunique()
)

Weather data saved to: ../data/cleaned/weather_data.csv
Rows: 50
Unique destinations: 50
